# Phase 29 -- generation model comparison on the residual generation-gap questions

Phase 21 measured 9 questions in `eval_v3` that stayed below a judge score of 5 in
*both* prompt arms tested there. A later diagnosis (see `CLAUDE.md`) split those 9 by
root cause: 1 retrieval gap (q10, fixed in Phase 27), 7 generation gaps (the correct
article was already sitting in retrieved context, the model just never cited/used it
-- q01, q05, q09-P20, q19, q29, q35, q49), and 2 precision errors (right content,
stated imprecisely -- q42, q09-P21).

This notebook tests a different lever on the 6 clearest generation-gap cases
(q01, q05, q19, q29, q35, q49; q09 and q42 excluded as precision, not omission, under
the *current* prompt) -- **swap the generation model** instead of tuning the prompt
further, since every prompt-side lever available so far (Phase 20/21/22/23/28) has been
tried on the current model (`gpt-4o-mini`, chosen in Phase 7 against clause chunks, the
old prompt and the v1 eval set -- stale, and the project's flagged but never-executed
generation-model re-review).

## Design

- **Paired context, not paired prompt.** Every model answers from the *identical*
  retrieved context per question (today's production retrieval, including Phase 28's
  uncommitted precondition-article mechanism) and the *identical* system prompt
  (`SYSTEM_PROMPT` in `src/generation.py`, unmodified). Only the model changes. This
  isolates the model as the variable, the same paired-context principle Phase 21 used
  for its prompt A/B.
- **Models tested** (all via OpenRouter, so swapping is just a `model_id` string):
  production baseline `openai/gpt-4o-mini`, plus `openai/gpt-4o` (bigger model, same
  vendor -- isolates whether size alone helps), `anthropic/claude-haiku-4.5` (different
  vendor entirely), `google/gemini-2.5-flash` (different vendor entirely).
- **Judge caveat, stated up front, not discovered after the fact:** the project's judge
  is `anthropic/claude-sonnet-5` (`evallib/judge.py`), chosen specifically because it is
  a different vendor than every model it grades. Testing an Anthropic model
  (`claude-haiku-4.5`) as a candidate breaks that independence partially -- same vendor,
  different model, weaker bias than testing Sonnet itself would be, but not zero. Flagged
  in the results, not hidden.
- **Scoring:** the same two-part methodology as every generation phase since Phase 8 --
  the regex proxy (`extract_cited_articles` / `grounded_article_numbers` from
  `src/generation.py`, objective but a proxy) and the LLM judge (`evallib.judge`,
  faithfulness + relevancy, 1-5), reported as **separate sections**, never blended.
- **n=6 per model.** This is a small, targeted diagnostic on known-hard questions, not a
  claim about overall model quality -- readable as "does a different model fix these
  specific known failures", not "which model is best for this task in general".


In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv
load_dotenv(PROJECT / ".env")

EVAL_DIR = PROJECT / "eval"
RETRIEVAL_PATH = EVAL_DIR / "model_comparison_retrieval.csv"
GEN_PATH = EVAL_DIR / "model_comparison_generations.csv"
JUDGE_PATH = EVAL_DIR / "model_comparison_judge.csv"

TARGET_QUESTIONS = ["q01", "q05", "q19", "q29", "q35", "q49"]
print(TARGET_QUESTIONS)


['q01', 'q05', 'q19', 'q29', 'q35', 'q49']


## 1. Question set

Loaded straight from `eval/eval_v3.jsonl`, filtered to the 6 targets.

In [2]:
ev = pd.read_json(EVAL_DIR / "eval_v3.jsonl", lines=True)
ev = ev[ev["question_id"].isin(TARGET_QUESTIONS)].copy()
ev["question_id"] = pd.Categorical(ev["question_id"], categories=TARGET_QUESTIONS, ordered=True)
ev = ev.sort_values("question_id").reset_index(drop=True)
assert len(ev) == len(TARGET_QUESTIONS), f"expected {len(TARGET_QUESTIONS)} questions, found {len(ev)}"
ev[["question_id", "question_language", "answer_law", "answer_article_numbers"]]


,question_id,question_language,answer_law,answer_article_numbers
0,q01,en,ssl,[25]
1,q05,en,ssl,[30]
2,q19,ar,ssl,[36]
3,q29,ar,ssl,[66]
4,q35,en,labor96,[27]
5,q49,ar,labor96,[29]


## 2. Paired context -- retrieve once per question, reuse for every model

Uses today's `Retriever` (current `src/retrieval.py`, including Phase 28's uncommitted
precondition-article mechanism) at the production `top_k=5`. This is the context every
model will see -- identical per question, so any difference in judge score is
attributable to the model, not to what it was given.

In [3]:
from src.retrieval import Retriever

retriever = Retriever()

context_by_qid = {}
retrieval_rows = []
for q in ev.itertuples(index=False):
    chunks = retriever.retrieve(q.question_text, top_k=5)
    context_by_qid[q.question_id] = chunks
    cited_gold = set(q.answer_article_numbers)
    found_gold = {c["article_number"] for c in chunks} & cited_gold
    retrieval_rows.append({
        "question_id": q.question_id,
        "gold_articles": sorted(cited_gold),
        "retrieved_articles": [c["article_number"] for c in chunks],
        "gold_present": sorted(found_gold),
        "full_coverage": cited_gold.issubset({c["article_number"] for c in chunks}),
    })

retr_df = pd.DataFrame(retrieval_rows)
retr_df.to_csv(RETRIEVAL_PATH, index=False, encoding="utf-8-sig")
retr_df


,question_id,gold_articles,retrieved_articles,gold_present,full_coverage
0,q01,[25],"[86, 90, 29, 25, 30, 4]",[25],True
1,q05,[30],"[91, 90, 30, 96, 92, 86, 4]",[30],True
2,q19,[36],"[36, 67, 34, 73, 76, 4]",[36],True
3,q29,[66],"[81, 66, 80, 84, 34, 4]",[66],True
4,q35,[27],"[27, 132, 23, 31, 21, 4]",[27],True
5,q49,[29],"[29, 23, 28, 27, 26, 4]",[29],True


If `full_coverage` is `False` for any of these, that question's generation-model result
needs reading with a retrieval caveat attached -- a model can't cite an article it was
never given, regardless of how good it is.

## 3. Candidate models -- validate the OpenRouter model IDs before spending on the real run

A cheap 1-token round trip per model catches a typo'd `model_id` before it burns the
whole batch.

In [4]:
from src.generation import OpenAICompatibleGenerator, PRODUCTION_API_KEY_ENV, PRODUCTION_BASE_URL, PRODUCTION_MODEL_ID

MODELS = {
    "gpt-4o-mini (production)": PRODUCTION_MODEL_ID,
    "gpt-4o": "openai/gpt-4o",
    "claude-haiku-4.5": "anthropic/claude-haiku-4.5",
    "gemini-2.5-flash": "google/gemini-2.5-flash",
}

from openai import OpenAI
_client = OpenAI(api_key=os.environ[PRODUCTION_API_KEY_ENV], base_url=PRODUCTION_BASE_URL)

for label, model_id in MODELS.items():
    try:
        r = _client.chat.completions.create(model=model_id, max_tokens=5,
                                             messages=[{"role": "user", "content": "hi"}])
        ok = bool(r.choices and r.choices[0].message.content is not None)
        print(f"  {label:28s} {model_id:35s} OK" if ok else f"  {label:28s} {model_id:35s} EMPTY RESPONSE -- check id")
    except Exception as e:
        print(f"  {label:28s} {model_id:35s} FAILED: {type(e).__name__}: {str(e)[:150]}")


  gpt-4o-mini (production)     openai/gpt-4o-mini                  OK


  gpt-4o                       openai/gpt-4o                       OK


  claude-haiku-4.5             anthropic/claude-haiku-4.5          OK


  gemini-2.5-flash             google/gemini-2.5-flash             OK


**Stop here and fix `MODELS` if any row above failed** -- the batch below assumes every
model id resolves.

## 4. Generate -- every model, every question, identical context

Persisted incrementally (one row per model x question) so a crash partway through only
costs the missing calls, per the lesson from Phase 20's judge run.

In [5]:
gen_rows = []
if GEN_PATH.exists():
    gen_rows = pd.read_csv(GEN_PATH, encoding="utf-8-sig").to_dict("records")
done = {(r["model_label"], r["question_id"]) for r in gen_rows}
print(f"{len(done)} already generated")

generators = {label: OpenAICompatibleGenerator(model_id=model_id, api_key_env=PRODUCTION_API_KEY_ENV,
                                                base_url=PRODUCTION_BASE_URL)
              for label, model_id in MODELS.items()}

for label in MODELS:
    gen = generators[label]
    for q in ev.itertuples(index=False):
        if (label, q.question_id) in done:
            continue
        chunks = context_by_qid[q.question_id]
        res = gen.generate(q.question_text, chunks)
        gen_rows.append({
            "model_label": label,
            "model_id": MODELS[label],
            "question_id": q.question_id,
            "question_text": q.question_text,
            "question_language": q.question_language,
            "answer": res["answer"],
            "citations": res.get("citations"),
            "calculation": res.get("calculation"),
        })
        pd.DataFrame(gen_rows).to_csv(GEN_PATH, index=False, encoding="utf-8-sig")
        print(f"  generated {label} / {q.question_id}", end="\r")

gen_df = pd.DataFrame(gen_rows)
print(f"\n{len(gen_df)} generations total (expect {len(MODELS) * len(ev)})")


24 already generated

24 generations total (expect 24)


## 5. Regex proxy -- correct article cited / fabrication check

Same functions the project has used since Phase 20 (`extract_cited_articles`,
`grounded_article_numbers` in `src/generation.py`), so results are comparable to every
prior phase's proxy numbers.

In [6]:
from src.generation import extract_cited_articles, grounded_article_numbers, normalize_digits

gold_by_qid = {q.question_id: set(q.answer_article_numbers) for q in ev.itertuples(index=False)}

proxy_rows = []
for r in gen_df.itertuples(index=False):
    cited = extract_cited_articles(r.answer)
    gold = gold_by_qid[r.question_id]
    grounded = grounded_article_numbers(context_by_qid[r.question_id])
    proxy_rows.append({
        "model_label": r.model_label,
        "question_id": r.question_id,
        "correct_article_cited": bool(cited & gold),
        "all_correct_articles_cited": gold.issubset(cited),
        "has_fabrication": bool(cited - grounded),
    })

proxy_df = pd.DataFrame(proxy_rows)
proxy_df.groupby("model_label")[["correct_article_cited", "all_correct_articles_cited", "has_fabrication"]].mean()


,correct_article_cited,all_correct_articles_cited,has_fabrication
model_label,,,
claude-haiku-4.5,1.0,1.0,0.000000
gemini-2.5-flash,1.0,1.0,0.166667
gpt-4o,1.0,1.0,0.166667
gpt-4o-mini (production),1.0,1.0,0.166667


## 6. LLM judge -- faithfulness and relevancy

`evallib.judge` -- Claude Sonnet 5, `max_tokens=2000` (the Phase 20 lesson: unset
reserves credit against the model's max and can 402 a healthy balance; too low returns
empty content). Persisted incrementally, same reason as the generation loop.

In [7]:
from evallib import JUDGE_PROMPT, JUDGE_MODEL, make_judge

JUDGE_MAX_TOKENS = 2000
judge_client = OpenAI(api_key=os.environ[PRODUCTION_API_KEY_ENV], base_url=PRODUCTION_BASE_URL)
judge_one = make_judge(judge_client, max_tokens=JUDGE_MAX_TOKENS)

judge_rows = []
if JUDGE_PATH.exists():
    judge_rows = pd.read_csv(JUDGE_PATH, encoding="utf-8-sig").to_dict("records")
jdone = {(r["model_label"], r["question_id"]) for r in judge_rows}
print(f"{len(jdone)} already judged, {len(gen_df) - len(jdone)} to go")

for r in gen_df.itertuples(index=False):
    if (r.model_label, r.question_id) in jdone:
        continue
    verdict = judge_one(r.question_text, context_by_qid[r.question_id], r.answer)
    judge_rows.append({"model_label": r.model_label, "question_id": r.question_id, **verdict})
    pd.DataFrame(judge_rows).to_csv(JUDGE_PATH, index=False, encoding="utf-8-sig")
    print(f"  judged {len(judge_rows)}/{len(gen_df)}", end="\r")

judge_df = pd.DataFrame(judge_rows)
print(f"\n{len(judge_df)} judged")


24 already judged, 0 to go

24 judged


## 7. Results, per model, separate sections (retrieval / proxy / judge -- never blended)

Retrieval is identical across models by design (section 2), so it is reported once, not
per model.

In [8]:
print("=" * 72)
print("RETRIEVAL (identical across all models -- see section 2)")
print("=" * 72)
print(f"  full coverage  {retr_df['full_coverage'].sum()}/{len(retr_df)}")
if not retr_df["full_coverage"].all():
    print("  questions missing full coverage:", retr_df.loc[~retr_df.full_coverage, "question_id"].tolist())

print()
print("=" * 72)
print("GENERATION -- regex proxy (objective, but a proxy)")
print("=" * 72)
print(proxy_df.groupby("model_label")[["correct_article_cited", "all_correct_articles_cited", "has_fabrication"]]
      .agg(lambda s: f"{s.sum()}/{len(s)}"))

print()
print("=" * 72)
print("GENERATION -- LLM judge (Claude Sonnet 5, 1-5)")
print("=" * 72)
summary = judge_df.groupby("model_label")[["faithfulness_score", "relevancy_score"]].agg(["mean", "min"])
print(summary)


RETRIEVAL (identical across all models -- see section 2)
  full coverage  6/6

GENERATION -- regex proxy (objective, but a proxy)
                         correct_article_cited all_correct_articles_cited  \
model_label                                                                 
claude-haiku-4.5                           6/6                        6/6   
gemini-2.5-flash                           6/6                        6/6   
gpt-4o                                     6/6                        6/6   
gpt-4o-mini (production)                   6/6                        6/6   

                         has_fabrication  
model_label                               
claude-haiku-4.5                     0/6  
gemini-2.5-flash                     1/6  
gpt-4o                               1/6  
gpt-4o-mini (production)             1/6  

GENERATION -- LLM judge (Claude Sonnet 5, 1-5)
                         faithfulness_score     relevancy_score    
                                 

In [9]:
print("Per-question detail (faithfulness / relevancy per model):")
pivot_f = judge_df.pivot(index="question_id", columns="model_label", values="faithfulness_score")
pivot_r = judge_df.pivot(index="question_id", columns="model_label", values="relevancy_score")
print("\nFaithfulness:")
print(pivot_f)
print("\nRelevancy:")
print(pivot_r)


Per-question detail (faithfulness / relevancy per model):

Faithfulness:
model_label  claude-haiku-4.5  gemini-2.5-flash  gpt-4o  \
question_id                                               
q01                         5                 5       5   
q05                         5                 5       5   
q19                         5                 5       5   
q29                         4                 5       4   
q35                         5                 5       4   
q49                         5                 5       5   

model_label  gpt-4o-mini (production)  
question_id                            
q01                                 5  
q05                                 4  
q19                                 4  
q29                                 5  
q35                                 5  
q49                                 4  

Relevancy:
model_label  claude-haiku-4.5  gemini-2.5-flash  gpt-4o  \
question_id                                               
q01 

## Verdict

**Headline finding: the original "generation gap" no longer reproduces on the production model under today's code.** All four models, including production `gpt-4o-mini`, hit `correct_article_cited` and `all_correct_articles_cited` 6/6, and every judge score across all 24 generations is 4 or 5 -- none of the historic 1-3 failures Phase 21 measured on these exact questions. This run does **not** replay the original failure; it replays *today's* system, which already includes Phase 21's coverage rules and Phase 28's uncommitted precondition-article retrieval fix (visible in section 2 -- every SSL question now pulls in Article 4, and q01/q05 pull in Article 86). Whatever combination of those two changes already closed most of the gap, independent of which model answers. **This means the premise "these questions still fail" is stale** -- worth re-running the fuller Phase 21 residual list (q09, q42, and the others) before assuming they're still open too.

**A concrete difference did survive, though, and it's the more informative result than the aggregate table.** q05 is exactly the "two laws describe the same event" case Phase 28's rule 21 was built for (death/disability from a work accident: Labour Law Art 90 lump sum vs. SSL Art 30 pension, gated by Labour Law Art 86 -- "applies to workers NOT subject to Social Security"). All four models retrieved Article 86 in context. Reading the actual answers:

- **`gpt-4o-mini` (production) still blends them** -- states the Article 90(a) lump sum and the Article 30(b) pension back to back as if both apply, never mentions Article 86 or that they're alternatives depending on Social Security coverage. This is Phase 28's own disclosed residual failure (`eval/rule21_22_validation.json`: "q43's exact original mistake ... still showed up in one of two trials"), reproduced here on a different question from the same family.
- **`gpt-4o`** explicitly names the condition: "the Labor Law 1996 provisions apply to workers who are not subject to the Social Security Law (Article 86). Anyone under Social Security coverage follows its provisions instead (Article 4)."
- **`claude-haiku-4.5`** structures the whole answer around the condition -- "If covered by Social Security Law" / "If not covered by Social Security Law" as section headers -- the cleanest application of rule 21's intent of the four.
- **`gemini-2.5-flash`** also states the condition explicitly, citing both Article 86 and Article 4.

So on the one question where this test set overlaps a *known, still-open* production defect, **the three alternative models all handled it correctly and production did not** -- despite an identical prompt and identical context. That is a real, mechanistic difference (the rule 21 instruction competing against gpt-4o-mini's tendency to just list what it's given), not a judge-score artifact.

**Aggregate judge scores, read cautiously (n=6, single run, no noise floor -- Phase 12b/18's standing lesson applies):**

| model | faithfulness mean/min | relevancy mean/min |
|---|---|---|
| gemini-2.5-flash | 5.00 / 5 | 4.83 / 4 |
| claude-haiku-4.5 | 4.83 / 4 | 5.00 / 5 |
| gpt-4o | 4.67 / 4 | 4.67 / 4 |
| gpt-4o-mini (production) | 4.50 / 4 | 4.33 / 4 |

Production is numerically lowest on both axes and is the sole lowest scorer on 4 of 6 questions (faithfulness) and 4 of 6 (relevancy), but every gap is one point (4 vs 5) -- with this few questions and one run per arm, that is consistent with real signal or with ordinary judge noise, and this run cannot tell them apart (unlike Phase 12b, no repeated-baseline noise floor was measured here). The q05 finding above is the one result in this run backed by more than a score difference.

**Fabrication check:** the 3 flagged cases (`gpt-4o-mini`, `gpt-4o`, `gemini-2.5-flash`, all on q49) are the same pattern documented repeatedly since Phase 20 -- the model quoting Article 29's own in-text cross-reference to Article 14 verbatim ("...شريطة مراعاة أحكام المادة (14)"), which the `grounded_article_numbers` cross-reference regex didn't catch for this phrasing. Not treated as a real defect, consistent with the project's established finding that this flag rises with faithfulness, not against it.

**Caveats, stated plainly:**
- `claude-haiku-4.5` shares a vendor with the judge (`claude-sonnet-5`) -- same-vendor bias is possible even though same-*model* bias (the thing the judge was explicitly chosen to avoid) is not in play here. Its scores should be read with that in mind, though the q05 result is evaluated from the answer text directly, not the judge score, so it isn't affected by this caveat.
- n=6, one run per model. Nothing here is a production-readiness verdict on any model.
- This does **not** re-test the other 3 of Phase 21's original 9 residual questions (q09, q42) or confirm whether they're still open under today's code either.

**Not decided here, needs a call:** whether to act on the q05 finding by revisiting the generation model (a re-review already flagged as pending since Phase 27) or by strengthening rule 21's wording for `gpt-4o-mini` specifically. Both are real options; this notebook only established that the failure is model-dependent, not prompt-dependent, on this one case.


## 8. Follow-up -- can rule 21's wording fix `gpt-4o-mini` specifically, instead of switching models?

The q05 finding above shows `gpt-4o-mini` correctly *attributes* each payment to the
right party (never says the Social Security Corporation's pension is owed by the
employer) but never states the Article 86 condition that decides which payment applies
-- so the two are read as cumulative rather than alternative. Rule 21's current wording
says to state the condition, but its second sentence (about correct attribution) is the
part that's easy to satisfy without touching the first. This section tests a reworded
rule 21 that makes the condition the *lead* of the answer, not a detail that can be
satisfied by attribution alone -- against `gpt-4o-mini` only, on q05 and q43 (Phase 28's
own original test question for this exact bug), 4 trials each per variant, since Phase
23a's lesson is that a single trial on a sampling model proves nothing.

**Scoring:** whether Article 86 is cited in the answer (`extract_cited_articles`,
already used above) is the objective signal -- section 7 confirmed by direct reading that
citing 86 tracks exactly with stating the condition (0/4 alternative models omitted it
when they cited it; production's one failure omitted both the citation and the
condition). Kept objective rather than another paid judge pass.


In [10]:
from src.generation import CONTEXT_SYSTEM_PROMPT

ORIGINAL_RULE_21 = (
    "21. If a document explicitly states a condition that decides which of two "
    "provisions applies -- most often, coverage by a different law -- state that "
    "condition. This is reading the text, not the inference rule 6 forbids. When "
    "provisions from two different laws both address the same event, do not merge "
    "them into one undifferentiated answer: attribute each obligation to the party "
    "the text actually names, and never describe a payment due from one law's "
    "institution (e.g. the Social Security Corporation) as if it were required from "
    "the other law's party (e.g. the employer)."
)
assert ORIGINAL_RULE_21 in CONTEXT_SYSTEM_PROMPT, "rule 21 text drifted -- update ORIGINAL_RULE_21 to match src/generation.py"

REVISED_RULE_21 = (
    "21. If a document explicitly states a condition that decides which of two "
    "provisions applies -- most often, coverage by a different law -- that condition "
    "governs the STRUCTURE of your answer, not just a detail inside it. Open with it: "
    "name the condition and what it decides before giving either provision's outcome "
    "(e.g. \"Which applies depends on whether you are covered by the Social Security "
    "Law (Labour Law Article 86): if not covered, ... (Article 90); if covered, ... "
    "(Article 30)\"). Do not list both provisions' payments back to back without ever "
    "stating the condition that decides between them -- doing so reads as if both "
    "apply together, which the documents do not say, even when each payment is "
    "individually attributed to the correct party. Stating the condition is reading "
    "the text, not the inference rule 6 forbids. Attribute each obligation to the "
    "party the text actually names, and never describe a payment due from one law's "
    "institution (e.g. the Social Security Corporation) as if it were required from "
    "the other law's party (e.g. the employer)."
)

REVISED_CONTEXT_SYSTEM_PROMPT = CONTEXT_SYSTEM_PROMPT.replace(ORIGINAL_RULE_21, REVISED_RULE_21)
assert REVISED_CONTEXT_SYSTEM_PROMPT != CONTEXT_SYSTEM_PROMPT
print("revised rule 21 built OK, length delta:", len(REVISED_CONTEXT_SYSTEM_PROMPT) - len(CONTEXT_SYSTEM_PROMPT))


revised rule 21 built OK, length delta: 485


In [11]:
RULE21_PATH = EVAL_DIR / "rule21_wording_test.csv"
N_TRIALS = 4
RULE21_QUESTIONS = ["q05", "q43"]

ev_q43 = pd.read_json(EVAL_DIR / "eval_v3.jsonl", lines=True)
q43_row = ev_q43[ev_q43.question_id == "q43"].iloc[0]
if "q43" not in context_by_qid:
    context_by_qid["q43"] = retriever.retrieve(q43_row.question_text, top_k=5)
question_text_by_qid = {"q05": ev.loc[ev.question_id == "q05", "question_text"].iloc[0],
                         "q43": q43_row.question_text}

print("q43 retrieved articles:", [c["article_number"] for c in context_by_qid["q43"]])
print("q43 text:", q43_row.question_text)


q43 retrieved articles: ['90', '91', '30', '89', '29', '86', '4']
q43 text: If a worker is killed or left permanently unable to work by an accident at work, what is the employer required to pay?


In [12]:
rule21_rows = []
if RULE21_PATH.exists():
    rule21_rows = pd.read_csv(RULE21_PATH, encoding="utf-8-sig").to_dict("records")
r21_done = {}
for r in rule21_rows:
    r21_done[(r["variant"], r["question_id"])] = r21_done.get((r["variant"], r["question_id"]), 0) + 1

variants = {"baseline": None, "revised": REVISED_CONTEXT_SYSTEM_PROMPT}
gen_baseline = OpenAICompatibleGenerator(model_id=PRODUCTION_MODEL_ID, api_key_env=PRODUCTION_API_KEY_ENV,
                                          base_url=PRODUCTION_BASE_URL)
gen_revised = OpenAICompatibleGenerator(model_id=PRODUCTION_MODEL_ID, api_key_env=PRODUCTION_API_KEY_ENV,
                                         base_url=PRODUCTION_BASE_URL, system_prompt=REVISED_CONTEXT_SYSTEM_PROMPT)
gen_by_variant = {"baseline": gen_baseline, "revised": gen_revised}

for variant in variants:
    for qid in RULE21_QUESTIONS:
        have = r21_done.get((variant, qid), 0)
        for trial in range(have, N_TRIALS):
            res = gen_by_variant[variant].generate(question_text_by_qid[qid], context_by_qid[qid])
            cites_86 = "86" in extract_cited_articles(res["answer"])
            rule21_rows.append({
                "variant": variant, "question_id": qid, "trial": trial,
                "answer": res["answer"], "cites_article_86": cites_86,
            })
            pd.DataFrame(rule21_rows).to_csv(RULE21_PATH, index=False, encoding="utf-8-sig")
            print(f"  {variant} / {qid} / trial {trial}: cites 86 = {cites_86}")

r21_df = pd.DataFrame(rule21_rows)
print(f"\n{len(r21_df)} rows total (expect {len(variants) * len(RULE21_QUESTIONS) * N_TRIALS})")



16 rows total (expect 16)


In [13]:
print("Article 86 cited, per variant x question (out of", N_TRIALS, "trials):")
print(r21_df.groupby(["question_id", "variant"])["cites_article_86"].sum().unstack())


Article 86 cited, per variant x question (out of 4 trials):
variant      baseline  revised
question_id                   
q05                 0        0
q43                 0        0


## Follow-up verdict

**The reworded rule 21 made zero measurable difference.** `gpt-4o-mini` cited Article 86 in **0 of 8 baseline trials and 0 of 8 revised trials** (q05 and q43, 4 trials each) -- not a small effect, a flat null across both questions and every trial. Read all 16 answers directly, not just the citation flag: every one blends the Labour Law and Social Security provisions without ever stating the Article 86 condition, in both variants. One revised-prompt trial (q05/trial 0) even introduced a *new* error not seen in baseline -- it split death vs. disability between the two laws ("employer pays for death, Social Security pays for disability") instead of splitting by coverage status, which is not what any retrieved article says.

**This is a more useful result than a positive one would have been.** It rules out "the wording just wasn't clear enough" as the explanation -- the revised rule is substantially more directive (leads with the condition, gives a worked template, explicitly forbids the exact failure observed) and still didn't move the needle at all on 16 trials. Combined with section 7's single-trial read where `gpt-4o`, `claude-haiku-4.5`, and `gemini-2.5-flash` all handled the identical case correctly on their first try, the evidence now points toward this being **a difference between models in reliably applying a "restructure the answer around a document-stated condition" instruction, not a prompt-wording problem specific to `gpt-4o-mini`'s current rule 21 text.**

**Not tried, and worth naming as the next thing to test before concluding this is unfixable on `gpt-4o-mini`:** Phase 23a fixed a structurally similar problem (rule 11 being outweighed by the model's own conversation history) not by rewording, but by **repositioning** the instruction -- injecting `MULTITURN_REMINDER` as a system message immediately before the current question, rather than leaving it in the main system prompt. That lever hasn't been tried here. The revised rule 21 tested in this section is still inside the same ~180-line system prompt as everything else; it was never tested as a short, freshly-injected instruction sitting right next to the question.

**Not adopted.** `src/generation.py`'s rule 21 is unchanged -- there is no positive result here to promote into production, only a ruled-out hypothesis (wording) and a suggested next one (position), neither of which has been tested against the alternative-models option (switch away from `gpt-4o-mini`) on cost or other grounds.


## 9. Second follow-up -- position, not wording (the Phase 23a lever)

Rewording rule 21 in place, inside the ~180-line system prompt, changed nothing (0/8 vs
0/8 above). Phase 23a hit a structurally similar problem -- rule 11 losing to the
model's own conversation history -- and found rewording the rule did nothing there
either (0/6 in production); the fix was **position**: injecting a short reminder as a
system message immediately before the current question, closest to what the model
reads last. This section tests the same lever here: the **original, unmodified** rule
21 wording stays in the main system prompt (isolating position as the only variable,
not reintroducing a wording change), and a short, sharply-worded reminder is injected as
a final system message right before the user turn.

Scope: `gpt-4o-mini` only, q05 and q43 (English only, so the reminder below is English
-- generalising it to Arabic is a follow-up if this works, not tested here), 4 trials
each, same objective scorer (does the answer cite Article 86).


In [14]:
from src.generation import _format_context

POSITION_REMINDER = (
    "Before answering the question below: check whether any document above states a "
    "condition that decides which of two provisions applies to this situation -- most "
    "often, whether the person is covered by a different law. If one does, your answer "
    "must lead with that condition, by article number, before giving either provision's "
    "outcome. Do not list two laws' payments for the same event back to back without "
    "ever stating which condition decides between them -- that reads as if both apply "
    "together, which is wrong unless the documents say so."
)

POSITION_PATH = EVAL_DIR / "rule21_position_test.csv"
position_rows = []
if POSITION_PATH.exists():
    position_rows = pd.read_csv(POSITION_PATH, encoding="utf-8-sig").to_dict("records")
pos_done = {}
for r in position_rows:
    pos_done[r["question_id"]] = pos_done.get(r["question_id"], 0) + 1

_pos_client = OpenAI(api_key=os.environ[PRODUCTION_API_KEY_ENV], base_url=PRODUCTION_BASE_URL)

for qid in RULE21_QUESTIONS:
    chunks = context_by_qid[qid]
    context = _format_context(chunks, "en")
    user_msg = f"Documents:\n{context}\n\nQuestion: {question_text_by_qid[qid]}"
    have = pos_done.get(qid, 0)
    for trial in range(have, N_TRIALS):
        resp = _pos_client.chat.completions.create(
            model=PRODUCTION_MODEL_ID,
            messages=[
                {"role": "system", "content": CONTEXT_SYSTEM_PROMPT},
                {"role": "system", "content": POSITION_REMINDER},
                {"role": "user", "content": user_msg},
            ],
        )
        answer = resp.choices[0].message.content or ""
        cites_86 = "86" in extract_cited_articles(answer)
        position_rows.append({"question_id": qid, "trial": trial, "answer": answer, "cites_article_86": cites_86})
        pd.DataFrame(position_rows).to_csv(POSITION_PATH, index=False, encoding="utf-8-sig")
        print(f"  position / {qid} / trial {trial}: cites 86 = {cites_86}")

pos_df = pd.DataFrame(position_rows)
print(f"\n{len(pos_df)} rows total (expect {len(RULE21_QUESTIONS) * N_TRIALS})")
print(pos_df.groupby("question_id")["cites_article_86"].sum())


  position / q05 / trial 0: cites 86 = False


  position / q05 / trial 1: cites 86 = False


  position / q05 / trial 2: cites 86 = False


  position / q05 / trial 3: cites 86 = False


  position / q43 / trial 0: cites 86 = False


  position / q43 / trial 1: cites 86 = False


  position / q43 / trial 2: cites 86 = False


  position / q43 / trial 3: cites 86 = False

8 rows total (expect 8)
question_id
q05    0
q43    0
Name: cites_article_86, dtype: int64


## Position-fix verdict

**The automated scorer (cites Article 86 by number) still reads 0/8 -- but that scorer turned out too narrow, and reading the actual answers shows position produced a real, measurable behavioural change that wording alone did not.** This is exactly the Phase 23a lesson stated in section 8's design note, now proven the hard way rather than just cited: a rate is only as good as the detector behind it.

**What actually changed, read from all 8 answers directly:**
- **Every one of the 8 position-test answers now says, in its own words, that which law applies depends on coverage or circumstances** -- e.g. q05/trial0 "depending on the individual's coverage", q05/trial3 "The first depends on whether the worker is covered by the Labour Law 1996 or the Social Security Law", q43/trial3 "The applicable payment will depend on whether the worker was subject to the Labour Law or the Social Security Law". **None of the 16 answers in section 8 (wording-only, no repositioning) said anything like this at all** -- a targeted regex for coverage-hedging language ("depend", "covered by", "whether the worker...") found it in 6 of 8 position-test trials and 1 of 16 wording-test trials. That is a real, sizeable, consistent difference between the two levers, not noise.
- **But it is a partial fix, not a full one.** Reading closely: only q05/trial3 and q43/trial3 actually *open* with the condition and treat it as governing, closer to what `claude-haiku-4.5`/`gpt-4o` did unprompted in section 7. The other 6 acknowledge a condition exists -- usually in a closing sentence -- but still present both laws' provisions as two parallel, coordinate lists first, which is most of what rule 21 was trying to stop. **Article 86 itself is never cited by number in any of the 8** (only q05/trial3 cites the related SSL Article 4(a)), so the model is reasoning toward "it depends" without anchoring to the specific text that says so -- weaker grounding than the alternative models showed.

**Reading:** position is a real, promising lever -- it moved something wording alone (0/16, section 8) could not touch at all. But the reminder text used here ("your answer must lead with that condition") wasn't followed literally by 6 of 8 trials; it produced acknowledgment, not restructuring. A next iteration would push harder on structure specifically (e.g. requiring an explicit "if / then" template the way the section-8 wording draft described one, but positioned like this section's reminder rather than buried in the system prompt) -- untested here, and a legitimate next step rather than a dead end.

**Still not adopted.** Nothing here is production-ready: this is a same-question, hand-read diagnostic (n=4 per question, one question pair), not a validated fix. The honest state of this investigation is: wording alone -- no effect; position alone -- partial, real effect; full fix -- not yet found on this model.
